# Three-Arm Block-Attention Ablation -- AR(1) Grid, C=84

CI vs CD vs CD_Block on the synthetic AR(1) family at C=84, rho=0.5. Tests whether restricting CD's cross-variate attention to a fixed partition changes accuracy or cost relative to full CD, on a family with no leader-follower structure to align groups to -- the partition here is contiguous groups of 3, declared neutral rather than structure-aligned.

Windowing is generated fresh from the committed AR(1) generator rather than matched to either of the two window-count regimes present in the original grid; the resulting count is measured and reported, not assumed.

In [ ]:
# -- Imports and device setup --------------------------------------------------
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import hashlib
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# -- Committed-module import with tripwire asserts ------------------------------
# Model code is not inlined here. It comes from the attached b1-block-attn
# dataset, the same committed models.py / models_cd_block.py used by the
# leader-follower block-attention ablation, verified by hash before use.
import sys

EXPECTED_FILES = {"models.py", "models_cd_block.py"}
AMP_FIX_MARKER = "Allocated from the first encoder output"

INPUT_ROOT = Path("/kaggle/input")
_hits = sorted(INPUT_ROOT.rglob("models_cd_block.py"))
assert len(_hits) == 1, (
    f"Expected exactly one attached dataset containing models_cd_block.py; found {_hits}. "
    "Attach b1-block-attn and nothing else that carries a models_cd_block.py."
)
MODULE_DIR = _hits[0].parent
_py_files = {p.name for p in MODULE_DIR.glob("*.py")}
assert _py_files == EXPECTED_FILES, (
    f"Dataset must contain exactly {sorted(EXPECTED_FILES)}; found {sorted(_py_files)}."
)
_src = (MODULE_DIR / "models_cd_block.py").read_text()
assert AMP_FIX_MARKER in _src, (
    "models_cd_block.py lacks the AMP-fix marker comment -- a pre-fix version was uploaded."
)

sys.path.insert(0, str(MODULE_DIR))
import models as M
import models_cd_block as MB

assert callable(getattr(M, "build_model", None)), "models.py must expose build_model()"
for _name in ("PatchTST_CD_Block", "contiguous_groups"):
    assert hasattr(MB, _name), f"models_cd_block.py must expose {_name}"
for _f in sorted(EXPECTED_FILES):
    _h = hashlib.sha256((MODULE_DIR / _f).read_bytes()).hexdigest()[:16]
    print(f"  {_f}: sha256[:16]={_h}")
print(f"Committed modules loaded from {MODULE_DIR}")


In [ ]:
# -- Experiment configuration ----------------------------------------------------
# AR(1) grid, C=84, rho=0.5 (single cell; the mid-correlation setting). Windowing
# is generated fresh from PHI/T_TOTAL/BURN_IN below rather than matched to
# either window-count regime in the original grid -- the resulting
# window count is measured and printed, not assumed.
#
# Architecture matches the standard synthetic-family protocol (Table 2), not
# the boundary-P4 tranche's P=4 variant, which answers a different question.

PHI:   float = 0.8
RHO:   float = 0.5
C:     int   = 84

MODES: list[str] = ["CI", "CD", "CD_Block"]
SEEDS: list[int] = [42, 123, 456, 789, 1011]

# AR(1) generator (matches the committed grid generator exactly).
T_TOTAL:    int   = 14_400
BURN_IN:    int   = 1_000
TRAIN_FRAC: float = 0.6
VAL_FRAC:   float = 0.2

# Sequence and patching (standard synthetic-family protocol).
SEQ_LEN:      int = 512
PRED_LEN:     int = 96
PATCH_SIZE:   int = 16
PATCH_STRIDE: int = 8

# Architecture (Table 2, Synthetic + ETTh1 config).
D_MODEL:  int   = 64
N_HEADS:  int   = 8
N_LAYERS: int   = 3
DROPOUT:  float = 0.2

# Training.
LR:            float = 1e-4
WEIGHT_DECAY:  float = 1e-4
MAX_EPOCHS:    int   = 50
PATIENCE:      int   = 10
GRAD_CLIP:     float = 1.0
WARMUP_EPOCHS: int   = 10

# Batch sizes: CI at the committed C=84 grid batch (32), keeping CI rows
# comparable to the existing grid protocol at this C. CD and CD_Block share
# the ablation-family committed batch 8 (the grid's own C=84 CD ran batch 1
# under a pre-fused-attention memory constraint that no longer applies).
# The ablation isolates attention scope, so both CD-family arms must run the
# identical protocol, step count included -- batch size is fixed, not
# auto-halved on OOM, to preserve that guarantee. The CI-vs-CD-family step
# asymmetry is the committed, disclosed one.
BATCH_BY_MODE: dict[str, int] = {"CI": 32, "CD": 8, "CD_Block": 8}

# Neutral partition: the AR(1) grid has no leader-follower structure to align
# groups to, so contiguous groups of 3 are used by construction, not because
# they align with anything in the data-generating process.
GROUP_SIZE: int = 3
GROUPS: list[list[int]] = MB.contiguous_groups(C, GROUP_SIZE)

N_PATCHES = (SEQ_LEN - PATCH_SIZE) // PATCH_STRIDE + 1
assert N_PATCHES == MB.num_patches(SEQ_LEN, PATCH_SIZE, PATCH_STRIDE)

# -- CFG hash guard: catches an edited config being run without re-deriving
# the expected hash, the same discipline the boundary-sweep notebooks use.
CFG: dict = {
    "PHI": PHI, "RHO": RHO, "C": C, "MODES": MODES, "SEEDS": SEEDS,
    "T_TOTAL": T_TOTAL, "BURN_IN": BURN_IN, "TRAIN_FRAC": TRAIN_FRAC, "VAL_FRAC": VAL_FRAC,
    "SEQ_LEN": SEQ_LEN, "PRED_LEN": PRED_LEN, "PATCH_SIZE": PATCH_SIZE, "PATCH_STRIDE": PATCH_STRIDE,
    "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS, "DROPOUT": DROPOUT,
    "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY, "MAX_EPOCHS": MAX_EPOCHS, "PATIENCE": PATIENCE,
    "GRAD_CLIP": GRAD_CLIP, "WARMUP_EPOCHS": WARMUP_EPOCHS, "BATCH_BY_MODE": BATCH_BY_MODE,
    "GROUP_SIZE": GROUP_SIZE, "GROUPS": GROUPS,
}
_cfg_hash = hashlib.sha256(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:16]
assert _cfg_hash == "456efc2019027eaf", (
    f"Embedded CFG hash {_cfg_hash} != expected '456efc2019027eaf' -- this "
    f"notebook's CFG literal was edited after generation; re-derive the expected hash "
    f"before trusting the run list below."
)

# -- Session budget (12h Kaggle cap kills a run WITH NO SAVED OUTPUT if unmanaged) --
SESSION_T0 = time.time()
# REQUIRED, set before Run All: this session's actual remaining Kaggle quota, in
# seconds. No numeric default is provided on purpose -- a stale or generic value
# here can exceed Kaggle's 12h hard cap, making the budget check below
# unreachable and killing the session with no saved output.
SESSION_BUDGET_S = None
assert SESSION_BUDGET_S is not None, (
    "SESSION_BUDGET_S is unset. Set it to this session's actual remaining Kaggle "
    "quota in seconds before Run All. Do not reuse a previous session's value or "
    "assume the full 12h cap is available."
)
_BUDGET_MARGIN_S = 900.0  # test eval + checkpoint + version-save overhead
# Conservative per-epoch seeds; refined in-place as real epochs complete.
EPOCH_EST_S = {"CI": 120.0, "CD": 400.0, "CD_Block": 400.0}

print(f"N_PATCHES={N_PATCHES}  C*N={C * N_PATCHES}  groups={len(GROUPS)} (size {GROUP_SIZE})  "
      f"total runs={len(MODES) * len(SEEDS)}  CFG hash={_cfg_hash}")


In [ ]:
# -- Data generator (matches the committed AR(1) grid generator exactly) --------

def generate_ar1(num_variates: int, phi: float, rho: float, seed: int) -> np.ndarray:
    """AR(1) process with compound-symmetry innovation covariance."""
    rng   = np.random.default_rng(seed)
    Sigma = np.full((num_variates, num_variates), rho, dtype=np.float64)
    np.fill_diagonal(Sigma, 1.0)
    L     = np.linalg.cholesky(Sigma)
    total = T_TOTAL + BURN_IN
    X     = np.zeros((total, num_variates), dtype=np.float64)
    for t in range(1, total):
        X[t] = phi * X[t - 1] + L @ rng.standard_normal(num_variates)
    return X[BURN_IN:]


def split_normalise(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """60/20/20 split with per-channel z-score fit on train only."""
    n_train = int(len(data) * TRAIN_FRAC)
    n_val   = int(len(data) * VAL_FRAC)
    train, val, test = data[:n_train], data[n_train:n_train + n_val], data[n_train + n_val:]
    mean = train.mean(axis=0, keepdims=True)
    std  = np.where(train.std(axis=0, keepdims=True) == 0, 1.0, train.std(axis=0, keepdims=True))
    return (train - mean) / std, (val - mean) / std, (test - mean) / std


def make_windows(data: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    """Sliding-window (x, y) pairs via stride tricks -- no Python loop."""
    T, Cv = data.shape
    n = T - SEQ_LEN - PRED_LEN + 1
    if n <= 0:
        raise ValueError(f"Not enough timesteps: {T}")
    s0, s1 = data.strides
    view = np.lib.stride_tricks.as_strided(
        data, shape=(n, SEQ_LEN + PRED_LEN, Cv), strides=(s0, s0, s1))
    xs = np.ascontiguousarray(view[:, :SEQ_LEN])
    ys = np.ascontiguousarray(view[:, SEQ_LEN:])
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)


def empirical_rho(train_data: np.ndarray) -> float:
    """Mean of upper-triangle pairwise Pearson correlations on the training split."""
    corr = np.corrcoef(train_data.T)
    i, j = np.triu_indices(corr.shape[0], k=1)
    return float(np.mean(corr[i, j]))


# Fresh generation, measured rather than assumed: print the real window
# count and empirical rho for this C/rho/seed combination before training
# starts, so any drift from the original grid's figures is visible up front.
_probe = generate_ar1(C, PHI, RHO, seed=SEEDS[0])
_tr, _va, _te = split_normalise(_probe)
_n_train_windows = len(_tr) - SEQ_LEN - PRED_LEN + 1
print(f"C={C} rho={RHO}: {len(_probe)} total timesteps, {_n_train_windows} train windows, "
      f"empirical_rho={empirical_rho(_tr):.4f} (seed={SEEDS[0]} probe)")
del _probe, _tr, _va, _te


In [ ]:
# -- Model construction from committed modules -----------------------------------

def build_arm(mode: str) -> nn.Module:
    if mode in ("CI", "CD"):
        return M.build_model(
            mode, seq_len=SEQ_LEN, pred_len=PRED_LEN, num_variates=C,
            patch_size=PATCH_SIZE, stride=PATCH_STRIDE, d_model=D_MODEL,
            n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)
    if mode == "CD_Block":
        return MB.PatchTST_CD_Block(
            groups=GROUPS, num_variates=C, seq_len=SEQ_LEN, pred_len=PRED_LEN,
            patch_size=PATCH_SIZE, stride=PATCH_STRIDE, d_model=D_MODEL,
            n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)
    raise ValueError(f"Unknown mode: {mode}")


def _n_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())


# -- Architecture assertions: equal capacity across arms before any training,
# so a difference in results is attributable to attention scope alone, not
# an accidental parameter-count mismatch between arms.
_x = torch.zeros(2, SEQ_LEN, C)
_models = {mode: build_arm(mode) for mode in MODES}
for _mode, _m in _models.items():
    _y = _m(_x)
    assert _y.shape == (2, PRED_LEN, C), f"{_mode} wrong output shape: {_y.shape}"
    assert _m.head.in_features == N_PATCHES * D_MODEL, f"{_mode} head wrong: {_m.head}"
    assert _m.head.out_features == PRED_LEN, f"{_mode} head wrong: {_m.head}"
    assert hasattr(_m, "encoder"), f"{_mode} lacks .encoder"

_p_cd, _p_blk = _n_params(_models["CD"]), _n_params(_models["CD_Block"])
assert _p_cd == _p_blk, f"Param mismatch: CD={_p_cd} CD_Block={_p_blk}"
assert _p_cd == _n_params(_models["CI"]), "CI/CD param mismatch"
print(f"Arms OK: {_p_cd:,} params each; head Linear({N_PATCHES * D_MODEL}, {PRED_LEN})")
print(f"CD_Block partition: {len(GROUPS)} groups of {GROUP_SIZE}")
del _models, _x, _y
free_cuda()


In [ ]:
# -- Training engine: per-epoch atomic checkpoint + resume ----------------------
# The whole-run-level resume used elsewhere in this project's block-attention
# family (skip/redo an entire run) is not used here: at this family's per-run
# cost, a session death mid-run would lose hours of progress rather than at
# most one epoch. This engine checkpoints every epoch instead, matching the
# boundary-sweep notebooks' resume contract.

def _atomic_torch_save(obj: dict, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _atomic_csv_save(rows: list[dict], path: Path) -> None:
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)


def _rng_capture() -> dict:
    return {
        "py": random.getstate(),
        "np": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def _rng_restore(st: dict) -> None:
    random.setstate(st["py"])
    np.random.set_state(st["np"])
    torch.set_rng_state(st["torch"].cpu() if torch.is_tensor(st["torch"]) else st["torch"])
    if st["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() if torch.is_tensor(t) else t for t in st["cuda"]])


def _cosine_warmup(optimizer: torch.optim.Optimizer, epoch: int, warmup: int) -> None:
    if epoch < warmup:
        lr = LR * (epoch + 1) / warmup
    else:
        progress = (epoch - warmup) / max(1, MAX_EPOCHS - warmup)
        lr = LR * 0.5 * (1.0 + np.cos(np.pi * progress))
    for g in optimizer.param_groups:
        g["lr"] = lr


@torch.no_grad()
def _evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        pred = model(xb.to(DEVICE)).cpu()
        mse += nn.functional.mse_loss(pred, yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred, yb, reduction="sum").item()
        n += yb.numel()
    return mse / n, mae / n


def run_one(mode: str, seed: int, results_dir: Path, ckpt_dir: Path,
            session_t0: float, session_budget_s: float, epoch_est_s: dict) -> dict | None:
    """One CI/CD/CD_Block run, resumable at epoch granularity.

    Returns the result row on completion, or None if the session budget was
    reached before the run finished -- the checkpoint stays on disk and the
    next session resumes from it automatically.
    """
    ckpt_path = ckpt_dir / f"ckpt_grid_C{C}_rho{RHO}_{mode.lower()}_s{seed}.pt"
    batch_size = BATCH_BY_MODE[mode]

    set_seed(seed)
    raw = generate_ar1(C, PHI, RHO, seed)
    tr, va, te = split_normalise(raw)
    x_tr, y_tr = make_windows(tr)
    x_va, y_va = make_windows(va)
    x_te, y_te = make_windows(te)
    emp_rho = empirical_rho(tr)

    train_dl = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=False)
    val_dl   = DataLoader(TensorDataset(x_va, y_va), batch_size=batch_size, shuffle=False, drop_last=False)
    test_dl  = DataLoader(TensorDataset(x_te, y_te), batch_size=batch_size, shuffle=False, drop_last=False)

    model     = build_arm(mode).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler    = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    criterion = nn.MSELoss()
    spe       = len(train_dl)

    start_epoch = 0
    stopped = False
    best_val, best_epoch, best_total_steps = float("inf"), 0, 0
    best_state_dict, no_improve, total_steps = None, 0, 0

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        _rng_restore(ckpt["rng"])
        best_val = ckpt["best_val"]
        best_epoch = ckpt["best_epoch"]
        best_total_steps = ckpt["best_total_steps"]
        best_state_dict = ckpt["best_model_state"]
        no_improve = ckpt["no_improve"]
        total_steps = ckpt["total_steps"]
        start_epoch = ckpt["epoch_done"]
        stopped = ckpt["stopped"]
        print(f"  [resume] {ckpt_path.name}: {start_epoch} epochs done, "
              f"best_val={best_val:.6f} @ epoch {best_epoch}, stopped={stopped}")

    print(f"[{mode} seed={seed}] params: {_n_params(model):,} | batch: {batch_size} | "
          f"steps/epoch: {spe} | start_epoch: {start_epoch + 1}")

    t0 = time.time()
    if not stopped:
        for epoch in range(start_epoch, MAX_EPOCHS):
            elapsed = time.time() - session_t0
            est = epoch_est_s[mode]
            if elapsed + est + _BUDGET_MARGIN_S > session_budget_s:
                print(f"  [{mode} seed={seed}] Session budget reached before epoch "
                      f"{epoch + 1}. Checkpoint saved; next session resumes here.")
                _atomic_torch_save({
                    "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "rng": _rng_capture(),
                    "best_val": best_val, "best_epoch": best_epoch,
                    "best_total_steps": best_total_steps, "best_model_state": best_state_dict,
                    "no_improve": no_improve, "total_steps": total_steps,
                    "epoch_done": epoch, "stopped": False,
                }, ckpt_path)
                del model, optimizer, scaler, criterion, train_dl, val_dl, test_dl
                free_cuda()
                return None

            ep_t0 = time.time()
            _cosine_warmup(optimizer, epoch, WARMUP_EPOCHS)
            model.train()
            for xb, yb in train_dl:
                optimizer.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                    loss = criterion(model(xb.to(DEVICE)), yb.to(DEVICE))
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                total_steps += 1

            val_mse, _ = _evaluate(model, val_dl)
            ep_s = time.time() - ep_t0
            epoch_est_s[mode] = max(epoch_est_s[mode], ep_s)

            improved = val_mse < best_val
            if improved:
                best_val, best_epoch, best_total_steps = val_mse, epoch + 1, total_steps
                best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            stop_now = no_improve >= PATIENCE

            if (epoch + 1) % 5 == 0 or epoch == 0 or stop_now:
                print(f"  [{mode} seed={seed}] Epoch {epoch + 1:3d}/{MAX_EPOCHS} | "
                      f"val MSE {val_mse:.4f} | best {best_val:.4f}@{best_epoch} | {ep_s:.0f}s/ep")

            _atomic_torch_save({
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "rng": _rng_capture(),
                "best_val": best_val, "best_epoch": best_epoch,
                "best_total_steps": best_total_steps, "best_model_state": best_state_dict,
                "no_improve": no_improve, "total_steps": total_steps,
                "epoch_done": epoch + 1, "stopped": stop_now,
            }, ckpt_path)

            if stop_now:
                print(f"  [{mode} seed={seed}] Early stop @ epoch {epoch + 1}. Best: epoch {best_epoch}.")
                break

    assert best_state_dict is not None, (
        f"[{mode} seed={seed}] No improving epoch was ever recorded -- likely "
        f"SESSION_BUDGET_S is too small to fit even one epoch plus margin."
    )
    model.load_state_dict(best_state_dict)
    test_mse, test_mae = _evaluate(model, test_dl)
    print(f"  [{mode} seed={seed}] Test MSE: {test_mse:.4f} | Test MAE: {test_mae:.4f} | "
          f"Best val: {best_val:.4f} @ epoch {best_epoch}")

    del model, optimizer, scaler, criterion, train_dl, val_dl, test_dl
    free_cuda()

    # Column set and order match the committed results_grid.csv schema exactly,
    # so these rows can sit alongside grid rows without schema reconciliation.
    return {
        "dataset": "synthetic_ar1", "C": C, "rho": RHO, "empirical_rho": round(emp_rho, 6),
        "mode": mode, "pred_len": PRED_LEN, "test_mse": round(test_mse, 6),
        "test_mae": round(test_mae, 6), "best_epoch": best_epoch, "seed": seed,
        "batch_size": batch_size, "steps_per_epoch": spe, "total_steps": best_total_steps,
    }


In [ ]:
# -- Main sweep: cross-session bootstrap, then run -------------------------------
RESULTS_DIR = Path("results")
CKPT_DIR = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
results_path = RESULTS_DIR / f"results_grid_C{C}_block_attn.csv"

# Cross-session bootstrap: a fresh session's /kaggle/working starts empty. If a
# previous version's output is attached as an input, seed working state from
# it -- registry first, then any checkpoints -- before deciding what still
# needs to run. Exactly one attached source is required for each; a second
# (whether a manually-attached dataset or the notebook's own auto-attached
# prior-kernel output) must be detached before Run All.
INPUT_ROOT = Path("/kaggle/input")
_prior_csvs = [p for p in INPUT_ROOT.rglob(results_path.name)]
if not results_path.exists() and _prior_csvs:
    assert len(_prior_csvs) == 1, (
        f"Multiple prior registries attached: {_prior_csvs}. Detach all but the "
        f"latest version's output.")
    shutil.copy(_prior_csvs[0], results_path)
    print(f"[bootstrap] registry seeded from {_prior_csvs[0]}")

_ckpt_glob = f"ckpt_grid_C{C}_rho{RHO}_*.pt"
_prior_ckpts: dict[str, list[Path]] = {}
for _p in INPUT_ROOT.rglob(_ckpt_glob):
    _prior_ckpts.setdefault(_p.name, []).append(_p)
for _name, _srcs in sorted(_prior_ckpts.items()):
    assert len(_srcs) == 1, (
        f"Checkpoint {_name} found in multiple attached inputs: {_srcs}. Detach all "
        f"but the latest version's output.")
    if not (CKPT_DIR / _name).exists():
        shutil.copy(_srcs[0], CKPT_DIR / _name)
        print(f"[bootstrap] {_name} seeded from {_srcs[0]}")

all_results = []
completed = set()
if results_path.exists() and results_path.stat().st_size > 0:
    existing = pd.read_csv(results_path)
    all_results = existing.to_dict("records")
    completed = {(r["mode"], int(r["seed"])) for r in all_results}
    print(f"Registry: {len(completed)}/{len(MODES) * len(SEEDS)} runs already complete.")

RUN_LIST = [(mode, seed) for mode in MODES for seed in SEEDS]

for mode, seed in RUN_LIST:
    key = (mode, seed)
    if key in completed:
        print(f"SKIP mode={mode} seed={seed} (already in results)")
        continue
    print(f"\n{'=' * 60}")
    print(f"Mode: {mode} | seed: {seed}")
    print(f"{'=' * 60}")
    result = run_one(mode, seed, RESULTS_DIR, CKPT_DIR, SESSION_T0, SESSION_BUDGET_S, EPOCH_EST_S)
    if result is None:
        print("Session budget reached. Next session: new version, attach THIS version's "
              "output as an input, Run All to resume.")
        break
    all_results.append(result)
    completed.add(key)
    _atomic_csv_save(all_results, results_path)
    print(f"  Results written ({len(all_results)}/{len(RUN_LIST)} runs complete)")

results_df = pd.DataFrame(all_results)
print("\n=== CI vs CD vs CD_Block -- AR(1) grid, C={} ===".format(C))
if not results_df.empty:
    print(results_df[["mode", "seed", "test_mse", "test_mae", "best_epoch", "steps_per_epoch"]].to_string(index=False))


In [ ]:
# -- Verify output files ----------------------------------------------------------
expected_rows = len(MODES) * len(SEEDS)
required = [results_path]
if not all(p.exists() for p in required):
    missing = [p for p in required if not p.exists()]
    raise RuntimeError(f"Output files missing: {missing}. Do not close the session.")

final_df = pd.read_csv(results_path)
print(f"Rows in {results_path.name}: {len(final_df)} (expected {expected_rows}: "
      f"{len(MODES)} modes x {len(SEEDS)} seeds)")
print(final_df[["mode", "seed", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))

if len(final_df) < expected_rows:
    print("WARNING: not all runs complete -- Run All again to resume from checkpoint.")
else:
    print("All output files verified.")
    print()
    print("=== Per-mode mean test MSE ===")
    print(final_df.groupby("mode")["test_mse"].agg(["mean", "std", "count"]).to_string())
